# Dropout Optimizations

* **Philox PRNG Architecture**: Salmon, J. K., Moraes, M. A., Dror, R. O., & Shaw, D. E. (2011). Parallel Random Numbers: As Easy as 1, 2, 3. In Proceedings of the International Conference for High Performance Computing, Networking, Storage and Analysis (pp. 1-12).
* **PyTorch PRNG Walkthrough:** Deshpande, A. (2023). *How PyTorch Generates Random Numbers*. Coding Confessions. https://blog.codingconfessions.com/p/how-pytorch-generates-random-numbers

This notebook draws theoretical concepts, constants, and round permutations from the original paper above, alongside practical insights on PyTorch's internal C++ implementation from Coding Confessions. Aside from `SpatialDropout`,`Dropout` is the only layer in this framework that implements an advanced custom kernel. While rewriting every layer at this level of hardware optimization is beyond the scope of this project, these two implementations serve as a targeted deep dive into high performance CUDA/HIP/C++ operations. 
The old implementation is shown below: 

In [ ]:
try:
    import cupy as cp
    xp = cp
except (ModuleNotFoundError, ImportError):
    import numpy as np
    xp = np 

class Dropout:
    def __init__(self, rate):
        #We write rate as the success rate. The dropout rate will then be 
        self.rate = 1 - rate
    
    def forward(self, inputs, training):
        #were gonna save the inputs and the binary mask
        self.inputs = inputs
        if not training:
            self.output = inputs.copy()
            return self.output
        self.binary_mask = xp.random.binomial(1, self.rate, size = inputs.shape) \
                        / self.rate
        self.output = self.binary_mask * self.inputs

        return self.output
    
    def backward(self, dvalues):
        self.dinputs = dvalues * self.binary_mask 

## Can we Really make any Improvements for Dropout? 

In general, with GPU programming there are two types of bounds to account for. 

* **Compute Bound**: Where the amount of FLOPs(floating point operations) of a compute core are saturated, meaning data has to wait for a cores thread to complete its task before moving on. 
* **Memory Bound**: Where the compute cores are waiting for data to be fetched from GPU VRAM. In this case, while GPUs benefit from holding information inside L1 and L2 cache, and perform calculations using the GPU registers. Lower level memory types can't account for slow VRAM, meaning compute cores idle waiting for new work. 

Our solution to memory bound, or more specifically memory bandwidth problems come from **kernel fusion** (`@cp.fuse()`, `cp.ElementwiseKernel`):  
  
We can circumvent this by avoiding unnecessary kernel operations, oftentimes combining operations (otherwise combining kernels) into a single or custom kernel that performs multiple operations at the same time. 
* A fused kernel will have data read from VRAM once, with operations being performed on the fly with the faster GPU registers and cache. Finally, we create a final tensor **once**, and write this back onto GPU VRAM. 

In our old implementation, we required 4 seperate kernel operations. Assume N elements means a 4D tensor:

    Step 1: tmp1 = xp.random.binomial(1, ...)                   ──► Launches Kernel 1 (Allocates N elements for tmp1)  
    Step 2: self.binary_mask = tmp1 / self.rate                 ──► Launches Kernel 2 (Allocates N elements for binary_mask)  
    Step 3: self.output = self.binary_mask * self.inputs        ──► Launches Kernel 3 (Allocates N elements for output)
This means we read from VRAM 3 times, and write to VRAM 3 times. We'll rewrite this pass such that we read from VRAM once, and write to VRAM once. We will be implementing the following:

    Step 1: self.output = _fused_dropout(inputs, self.rate)     ──► Launches Kernel 1 (Allocates N elements ONLY for output)

Normally I would discuss the use of the decorator `@cp.fuse()` which will combine out of place operations **or** out of place reductions into a fused kernel. However, for this case we'll use the `cp.ElementwiseKernel()` to create a custom kernel and create this fused dropout layer. For this, although we have a division by a scaler, then an elementwise multiplication operation after, we cannot combine these operations with `cp.random.binomial` as this is **not** a universal function. To recap, `@cp.fuse()` works with universal functions only (+, -, *, /) or `cp.add()`, `cp.multiply()`, `cp.maximum`, etc. in either elementwise or reduction operations, but does not work outside of these bounds. 



# How will we Code a Custom Kernel then? 
Looking at our code, we want to create a binary mask (a tensor of only 0s and 1s) and divide this by our rate `self.rate`. The simple reason for this is to amplify the remaining neurons such that the expected value of the tensor remains the same. Next, we'll perform an elementwise multiplication between the binary mask and our inputs, resulting in our desired `self.output`. 



The hard part will be creating our own `cp.random.binomial()` function that will we'll apply onto our inputs. The division and elementwise multiplication steps are trivial. 

Looking at how the other deep learning frameworks write their dropout layer, they avoid a costly random generation by using **Philox4x32-10 PRNG**, a counter-based psudo random number generator that uses an integer counter for the inital state along with mulitplication based mixing. Normal PRNGs mutate an internal state sequentially, ($S_{t+1} = f(S_t))$, which is costly for a GPU.

### How does Philox Work? 

For **Philox-4x32**, we'll contain 4 32-bit integers at a time. We'll have a 128-bit counter (GPUs read from VRAM in 128 bit contigous blocks per memory transaction), and produces a 128-bit output (four 32-bit random numbers). This means we generate 4 random numbers at once per memory transaction. There are 10 rounds with an order shown below. 

* Four 32-bit words  $[c_0, c_1, c_2, c_3]$.
* Key (K): Two 32-bit words $[K_0, K_1]$.
* Multipliers ($M_0, M_1$): Hardcoded 32 bit integers that satisfied high spectral test scores, with no short bit patterns, and odd hex digits. They do their job, and they do their job well. 
$$M_0 = \text{0xD2511F53}, \quad M_1 = \text{0xCD9E8D57}$$

Each round we perform 2 multiplications: 
$$
    \text{prod}_0 = M_0 \times c_0, \quad \text{prod}_1 = M_1 \times c_2
$$
With this, we arrive at two 64-bit results (prod). Inside each prod, we split the bit values into **High 32 bits** (the bits on the right), and **Low 32 bits** (the bits on the left). Next, the new state words for the next round ($c'$) are computed by combining the product halves with the state word and the key. 
$$
h_0 = \text{hi}(\text{prod}_0)) \oplus c_1 \oplus K_0
$$
$$
h_1 = \text{hi}(\text{prod}_1)) \oplus c_3 \oplus K_1
$$

Now we can rearrange the values for the next round. The output for a round become
$$
(c'_0, c'_, c'_2, c'_3) = (\text{lo}(\text{prod}_0, h_1, \text{lo}(\text{prod}_1), h_0))
$$
The values have now been shuffled around, the low parts of the products arrive at positions 0 and 2, while the XORed high parts sit at position 1 and 3.

#### After Each Round

Before the final 10th round, we'll update the keys themselves by two constants 

$$
K'_0 = K_0 + w_0, \quad K'_1 = K_1 + w_1
$$
where:
* $w_0$: 0x9E3779B9
* $w_1$: 0xBB67AE85

From the paper, these are the Weyl sequence constants 

hi

In [ ]:
from aether.config import config 

_philox_dropout_forward = None
_philox_dropout_backward = None

try:
    import cupy as cp
    xp = cp
except (ModuleNotFoundError, ImportError):
    import numpy as np
    xp = np 


# ----------------------------------------------------------------------
# Shared device-side Philox4x32-10 -> uniform float, used by both the
# forward and backward kernels so the exact same bits get regenerated.
# ----------------------------------------------------------------------
_PHILOX_PREAMBLE = r'''
__device__ __forceinline__ float philox_uniform(
        unsigned long long philox_seed,
        unsigned long long philox_offset,
        long long idx) {
 
    // 128-bit counter = {offset_lo, offset_hi, idx_lo, idx_hi}
    unsigned int c0 = (unsigned int)(philox_offset & 0xffffffffULL);
    unsigned int c1 = (unsigned int)(philox_offset >> 32);
    unsigned int c2 = (unsigned int)((unsigned long long)idx & 0xffffffffULL);
    unsigned int c3 = (unsigned int)((unsigned long long)idx >> 32);
 
    // 64-bit key = {seed_lo, seed_hi}
    unsigned int k0 = (unsigned int)(philox_seed & 0xffffffffULL);
    unsigned int k1 = (unsigned int)(philox_seed >> 32);
 
    #pragma unroll
    for (int round = 0; round < 10; round++) {
        // 32x32 -> 64 multiply, split into hi/lo. Plain arithmetic (no
        // __umulhi) so this compiles under both NVRTC and HIPRTC.
        unsigned long long p0 = (unsigned long long)0xD2511F53u * c0;
        unsigned long long p1 = (unsigned long long)0xCD9E8D57u * c2;
        unsigned int lo0 = (unsigned int)(p0 & 0xffffffffULL);
        unsigned int hi0 = (unsigned int)(p0 >> 32);
        unsigned int lo1 = (unsigned int)(p1 & 0xffffffffULL);
        unsigned int hi1 = (unsigned int)(p1 >> 32);
 
        unsigned int nc0 = hi1 ^ c1 ^ k0;
        unsigned int nc1 = lo1;
        unsigned int nc2 = hi0 ^ c3 ^ k1;
        unsigned int nc3 = lo0;
 
        c0 = nc0; c1 = nc1; c2 = nc2; c3 = nc3;
        k0 += 0x9E3779B9u;
        k1 += 0xBB67AE85u;
    }
 
    // top 24 bits of c0 -> uniform float in [0, 1), same convention
    // curand/PyTorch/TF use for uint32 -> uniform float.
    return (c0 >> 8) * (1.0f / 16777216.0f);
}
'''
 
_philox_dropout_forward = None
_philox_dropout_backward = None
 
if cp is not None:
    # inverted dropout, fully fused: one read of x, one write of y, no
    # intermediate mask array ever allocated.
    _philox_dropout_forward = cp.ElementwiseKernel(
        'T x, uint64 philox_seed, uint64 philox_offset, float64 keep_prob',
        'T y',
        '''
        float u = philox_uniform(philox_seed, philox_offset, i);
        y = (u < keep_prob) ? (T)(x / keep_prob) : (T)0;
        ''',
        'philox_dropout_forward',
        preamble=_PHILOX_PREAMBLE,
    )
 
    # regenerates the identical mask from the same (seed, offset) instead
    # of reading a stored mask tensor: one read of dvalues, one write of
    # dinputs, zero mask storage.
    _philox_dropout_backward = cp.ElementwiseKernel(
        'T dvalues, uint64 philox_seed, uint64 philox_offset, float64 keep_prob',
        'T dinputs',
        '''
        float u = philox_uniform(philox_seed, philox_offset, i);
        dinputs = (u < keep_prob) ? (T)(dvalues / keep_prob) : (T)0;
        ''',
        'philox_dropout_backward',
        preamble=_PHILOX_PREAMBLE,
    )

class Dropout(Layer):
    # each dropout instance recieves a unique stream ID
    _stream_counter = 0

    def __init__(self, rate, seed=None):
        super().__init__(seed=seed) # from Layer.__init__
        self.rate = 1 - rate

        stream_id = Dropout._stream_counter
        Dropout._stream_counter += 1
        self._seed = _derive_stream_seed(self.seed, stream_id)
        self._call_counter = 0 # bumped once per training-mode forward call

    def _forward_gpu(self, inputs, training):
        if not training:
            self.output = inputs.copy()
            return self.output

        offset = self._call_counter
        self._call_counter += 1 
        self._offset = offset   # stashed so _backward_gpu can regenerate mask

        self.output = _philox_dropout_forward(
            inputs, self._seed, offset, float(self.rate)
        )

        return self.output
    
    def _forward_fallback(self, inputs, training):
        if not training:
            self.output = inputs.copy()
            return self.output
        
        xp = config.get_array_module(inputs)
        self.binary_mask = xp.random.binomial(1, self.rate, size=inputs.shape) / self.rate
        self.output = self.binary_mask * self.inputs
        return self.output
    
    def _backward_gpu(self, dvalues):
        self.dinputs = _philox_dropout_backward(
            dvalues, self._seed, self._offset, float(self.rate)
        )
        return self.dinputs
    
    def _backward_fallback(self, dvalues):
        self.dinputs = dvalues * self.binary_mask
        return self.dinputs
    
    def _compile_for_device(self, device):
        """Triggered by Model.to(device) to map low-level hardware paths."""
        if device == 'cupy' and _philox_dropout_forward is not None and _philox_dropout_backward is not None:
            self.forward = self._forward_gpu
            self.backward = self._backward_gpu
        else:
            self.forward = self._forward_fallback
            self.backward = self._backward_fallback
